# 01 Feature Engineering, EDA and CNN Backbone Screening


**Pipeline:**

1. load the 42 candidate engineered predictors produced by `00_grid_resolution_selection.ipynb`;
2. perform target and predictor EDA before feature removal;
3. inspect missingness, sparsity, correlations, VIIRS NTL behaviour, clipped cells and Sentinel extremes;
4. audit multicollinearity and deterministic redundancy;
5. remove six redundant / low-information variables and save the final **36-feature** matrix to the original path;
6. extract frozen ImageNet ResNet18 / ResNet34 / ResNet50 embeddings;
7. screen CNN backbones with Random 5-fold and KMeans K=4 spatial validation;
8. preserve the ImageNet ResNet50 raw embedding file used by the later controlled multimodal experiment.

The spatial/raster aggregation itself is already carried out in `00_grid_resolution_selection.ipynb`; this notebook audits and finalises the selected 500 m feature matrix rather than rebuilding the entire GIS overlay pipeline.

The raw target is not clipped, transformed or filtered here. `log1p` is used only as an EDA diagnostic.

## 1. Imports, paths and feature definitions

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from matplotlib.colors import Normalize, ListedColormap
from matplotlib.cm import ScalarMappable
import matplotlib.patches as mpatches

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor

import rasterio
from rasterio.mask import mask as rio_mask

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights, ResNet34_Weights, ResNet50_Weights

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

FEATURE_DIR = PROJECT_DATA_DIR / "grid_size_selection" / "features"
GRID_DIR = PROJECT_DATA_DIR / "grid_size_selection" / "grids"

FEATURE_MATRIX_PATH = FEATURE_DIR / "feature_matrix_500m.csv"
GRID_PATH = GRID_DIR / "grid_features_500m.gpkg"
CANDIDATE_BACKUP_PATH = FEATURE_DIR / "feature_matrix_500m_candidate_preclean.csv"

OUTPUT_DIR = PROJECT_DATA_DIR / "eda_500m"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_RESULT_DIR = PROJECT_DATA_DIR / "grid_size_selection" / "model_comparison_500m"

for folder in [FEATURE_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "elec_consumption"

META_COLUMNS = [
    "grid_id", "centroid_x", "centroid_y",
    "elec_consumption", "elec_point_count", "elec_std",
]

CANDIDATE_FEATURES = [
    "area_m2",
    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",
    "NDVI", "NDBI", "brightness",
    "NTL_mean", "NTL_std", "NTL_max",
    "dist_major_road",
    "road_length_total", "road_count", "road_length_major",
    "road_density", "major_road_ratio",
    "poi_economic_count", "poi_social_count", "poi_other_count",
    "poi_total_count", "poi_shannon",
    "dist_economic_poi", "dist_social_poi",
    "pop_total", "pop_density_km2",
    "building_count", "building_area_total", "building_area_mean",
    "building_area_std", "building_coverage",
]

FEATURES_TO_DROP = [
    "brightness",
    "poi_total_count",
    "pop_total",
    "road_length_total",
    "building_area_total",
    "NTL_std",
]

FINAL_FEATURES = [
    "area_m2",
    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",
    "NDVI", "NDBI",
    "NTL_mean", "NTL_max",
    "dist_major_road",
    "road_count", "road_length_major", "road_density", "major_road_ratio",
    "poi_economic_count", "poi_social_count", "poi_other_count",
    "poi_shannon", "dist_economic_poi", "dist_social_poi",
    "pop_density_km2",
    "building_count", "building_area_mean", "building_area_std", "building_coverage",
]

# EDA cells below use this name; at this stage it deliberately means all candidates.
BASELINE_FEATURES = CANDIDATE_FEATURES

assert len(CANDIDATE_FEATURES) == 42
assert len(FEATURES_TO_DROP) == 6
assert len(FINAL_FEATURES) == 36
assert set(FINAL_FEATURES) == set(CANDIDATE_FEATURES) - set(FEATURES_TO_DROP)

for path in [FEATURE_MATRIX_PATH, GRID_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Computation device:", DEVICE)
print("Candidate feature count:", len(CANDIDATE_FEATURES))
print("Final feature count:", len(FINAL_FEATURES))
print("Main matrix path:", FEATURE_MATRIX_PATH)
print("EDA output:", OUTPUT_DIR)

## 2. Load and align the candidate 500 m dataset

If `00_grid_resolution_selection.ipynb` has just been rerun, the main CSV contains all 42 candidate variables and the candidate backup is refreshed. If this notebook is rerun after it has already written the final 36-feature CSV, the backup is used so the complete EDA remains reproducible.

In [ ]:
current_matrix = pd.read_csv(FEATURE_MATRIX_PATH)

if all(feature in current_matrix.columns for feature in CANDIDATE_FEATURES):
    df = current_matrix.copy()
    df.to_csv(CANDIDATE_BACKUP_PATH, index=False)
    candidate_source = "feature_matrix_500m.csv (candidate backup refreshed)"
elif CANDIDATE_BACKUP_PATH.exists():
    df = pd.read_csv(CANDIDATE_BACKUP_PATH)
    candidate_source = "feature_matrix_500m_candidate_preclean.csv"
else:
    missing = [feature for feature in CANDIDATE_FEATURES if feature not in current_matrix.columns]
    raise KeyError(
        "The main matrix is already cleaned but no candidate backup exists. "
        "Rerun 00_grid_resolution_selection.ipynb first. Missing candidate features: "
        f"{missing}"
    )

required_columns = META_COLUMNS + CANDIDATE_FEATURES
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise KeyError(f"Missing required candidate columns: {missing_columns}")

if df["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid_id values found in candidate matrix.")

grid_gdf = gpd.read_file(GRID_PATH)

if "grid_id" not in grid_gdf.columns:
    raise KeyError("grid_id is missing from grid geometry.")

if grid_gdf["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid_id values found in grid geometry.")

eda_gdf = (
    grid_gdf[["grid_id", "geometry"]]
    .merge(df, on="grid_id", how="inner", validate="one_to_one")
)

if len(eda_gdf) != len(df):
    raise ValueError("Some candidate rows have no matching grid geometry.")

print("Candidate source:", candidate_source)
print("Feature-matrix rows:", len(df))
print("Feature-matrix columns:", df.shape[1])
print("Spatially aligned rows:", len(eda_gdf))
print("CRS:", eda_gdf.crs)

display(df[META_COLUMNS + CANDIDATE_FEATURES[:8]].head())

## 3. Derived-feature engineering consistency

The derived indicators were constructed in `00_grid_resolution_selection.ipynb`. This check verifies the formulas after the 500 m resolution has been selected; it does not recompute the underlying GIS overlays.

In [ ]:
expected_features = {
    "NDVI": (
        (df["B08_mean"] - df["B04_mean"])
        / (df["B08_mean"] + df["B04_mean"] + 1e-9)
    ),
    "NDBI": (
        (df["B11_mean"] - df["B08_mean"])
        / (df["B11_mean"] + df["B08_mean"] + 1e-9)
    ),
    "brightness": df[["B02_mean", "B03_mean", "B04_mean", "B08_mean"]].mean(axis=1),
    "poi_total_count": df[["poi_economic_count", "poi_social_count", "poi_other_count"]].sum(axis=1),
    "road_density": (df["road_length_total"] / 1000) / (df["area_m2"] / 1_000_000),
    "major_road_ratio": df["road_length_major"] / (df["road_length_total"] + 1e-9),
    "pop_density_km2": df["pop_total"] / (df["area_m2"] / 1_000_000),
    "building_coverage": df["building_area_total"] / (df["area_m2"] + 1e-9),
}

rows = []
for feature, expected in expected_features.items():
    observed = pd.to_numeric(df[feature], errors="coerce")
    difference = (observed - expected).abs()
    rows.append({
        "feature": feature,
        "max_abs_difference": difference.max(),
        "mean_abs_difference": difference.mean(),
    })

derived_feature_consistency = pd.DataFrame(rows)
derived_feature_consistency.to_csv(TABLE_DIR / "derived_feature_consistency.csv", index=False)
display(derived_feature_consistency.round(10))

## 3. Data integrity and missingness

This cell checks whether modelling variables contain missing, infinite, duplicate or constant values before any model is fitted.

In [ ]:
numeric_columns = [
    TARGET_COLUMN
] + BASELINE_FEATURES

integrity_rows = []

for column in numeric_columns:
    values = pd.to_numeric(
        df[column],
        errors="coerce",
    )

    finite_values = values.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    integrity_rows.append({
        "variable": column,
        "missing_n": finite_values.isna().sum(),
        "missing_pct": finite_values.isna().mean() * 100,
        "unique_n": finite_values.nunique(dropna=True),
        "zero_n": (finite_values == 0).sum(),
        "zero_pct": (finite_values == 0).mean() * 100,
        "min": finite_values.min(),
        "max": finite_values.max(),
    })

integrity_table = pd.DataFrame(
    integrity_rows
)

integrity_table["constant_flag"] = (
    integrity_table["unique_n"] <= 1
)

display(
    integrity_table
    .sort_values(
        ["missing_pct", "zero_pct"],
        ascending=False,
    )
    .round(3)
)

integrity_table.to_csv(
    TABLE_DIR
    / "data_integrity_summary.csv",
    index=False,
)

print(
    "Duplicate grid_id:",
    int(df["grid_id"].duplicated().sum()),
)

print(
    "Constant variables:",
    integrity_table.loc[
        integrity_table["constant_flag"],
        "variable",
    ].tolist(),
)

## 4. Observed electricity-consumption distribution

The target distribution is examined before modelling. The 2nd and 98th percentiles are reported because the target contains both low and high tails.

In [ ]:
target = df[
    TARGET_COLUMN
].dropna()

target_percentiles = target.quantile(
    [
        0.00,
        0.01,
        0.02,
        0.05,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.98,
        0.99,
        1.00,
    ]
)

target_summary = pd.DataFrame({
    "statistic": [
        "count",
        "mean",
        "std",
        "skewness",
        "minimum",
        "p01",
        "p02",
        "p05",
        "p25",
        "median",
        "p75",
        "p90",
        "p95",
        "p98",
        "p99",
        "maximum",
    ],
    "value": [
        target.count(),
        target.mean(),
        target.std(),
        target.skew(),
        target_percentiles.loc[0.00],
        target_percentiles.loc[0.01],
        target_percentiles.loc[0.02],
        target_percentiles.loc[0.05],
        target_percentiles.loc[0.25],
        target_percentiles.loc[0.50],
        target_percentiles.loc[0.75],
        target_percentiles.loc[0.90],
        target_percentiles.loc[0.95],
        target_percentiles.loc[0.98],
        target_percentiles.loc[0.99],
        target_percentiles.loc[1.00],
    ],
})

display(
    target_summary.round(3)
)

target_summary.to_csv(
    TABLE_DIR
    / "target_distribution_summary.csv",
    index=False,
)

q02 = target.quantile(0.02)
q98 = target.quantile(0.98)

print(
    "Observed < 2nd percentile:",
    int((target < q02).sum()),
)

print(
    "Observed > 98th percentile:",
    int((target > q98).sum()),
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

sns.histplot(
    target,
    bins=45,
    kde=True,
    ax=axes[0],
)

axes[0].axvline(
    q02,
    linestyle="--",
    linewidth=1,
    label=f"2nd percentile = {q02:.1f}",
)

axes[0].axvline(
    q98,
    linestyle="--",
    linewidth=1,
    label=f"98th percentile = {q98:.1f}",
)

axes[0].set_title(
    "Observed electricity-consumption distribution"
)

axes[0].set_xlabel(
    "Electricity consumption"
)

axes[0].set_ylabel(
    "Grid count"
)

axes[0].legend()

sns.boxplot(
    x=target,
    ax=axes[1],
)

axes[1].set_title(
    "Observed electricity-consumption boxplot"
)

axes[1].set_xlabel(
    "Electricity consumption"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "target_distribution_and_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Raw target versus `log1p` target

This is a diagnostic only. It shows how much `log1p` reduces target skewness; it does not change the target used by later notebooks. Whether to use a transformed target must be decided by cross-validated predictive performance on the original scale.

In [ ]:
target_log = np.log1p(
    target
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

sns.histplot(
    target,
    bins=45,
    kde=True,
    ax=axes[0],
)

axes[0].set_title(
    f"Raw target | skew = {target.skew():.2f}"
)

axes[0].set_xlabel(
    "Electricity consumption"
)

sns.histplot(
    target_log,
    bins=45,
    kde=True,
    ax=axes[1],
)

axes[1].set_title(
    f"log1p target | skew = {target_log.skew():.2f}"
)

axes[1].set_xlabel(
    "log1p(electricity consumption)"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "target_raw_vs_log1p_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Raw skewness:",
    round(target.skew(), 3),
)

print(
    "log1p skewness:",
    round(target_log.skew(), 3),
)

## 6. Observed electricity spatial pattern

The colour scale is clipped to the observed 2nd–98th percentile only for visualisation. The underlying target values are unchanged.

In [ ]:
target_norm = Normalize(
    vmin=q02,
    vmax=q98,
    clip=True,
)

fig, ax = plt.subplots(
    figsize=(9, 8),
)

eda_gdf.plot(
    column=TARGET_COLUMN,
    ax=ax,
    cmap="viridis",
    norm=target_norm,
    edgecolor="none",
)

sm = ScalarMappable(
    norm=target_norm,
    cmap="viridis",
)

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.78,
    extend="both",
)

cbar.set_label(
    "Observed electricity consumption"
)

ax.set_title(
    "Observed electricity consumption at 500 m"
)

ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "observed_electricity_map_2_98pct.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 7. Spatial locations of the low and high tails

Only the tail grids are filled. All other grids are shown as light outlines.

In [ ]:
low_tail = (
    eda_gdf[
        eda_gdf[TARGET_COLUMN] < q02
    ]
    .copy()
)

high_tail = (
    eda_gdf[
        eda_gdf[TARGET_COLUMN] > q98
    ]
    .copy()
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
)

for ax in axes:
    eda_gdf.boundary.plot(
        ax=ax,
        color="#e6e6e6",
        linewidth=0.25,
    )
    ax.set_axis_off()

low_tail.plot(
    column=TARGET_COLUMN,
    ax=axes[0],
    cmap="Blues",
    edgecolor="none",
    legend=True,
    legend_kwds={
        "label": "Observed electricity consumption",
        "shrink": 0.75,
    },
)

axes[0].set_title(
    "Low-tail grids: Observed < 2nd percentile"
)

high_tail.plot(
    column=TARGET_COLUMN,
    ax=axes[1],
    cmap="Reds",
    edgecolor="none",
    legend=True,
    legend_kwds={
        "label": "Observed electricity consumption",
        "shrink": 0.75,
    },
)

axes[1].set_title(
    "High-tail grids: Observed > 98th percentile"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "observed_low_high_tail_locations.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Low-tail grids:", len(low_tail))
print("High-tail grids:", len(high_tail))

## 8. Predictor distribution, variance and skewness diagnostics

All tabular predictors are checked, not only the target. The table reports standard deviation, interquartile range, skewness, zero share and robust spread. Variables with almost no variation are flagged because they carry little information for prediction.

The dissertation does not need to show 37 separate distribution plots; the complete diagnostic table can be retained as supporting material while representative plots are used in the main text.

In [ ]:
predictor_rows = []

for column in BASELINE_FEATURES:
    series = pd.to_numeric(
        df[column],
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr = q75 - q25

    predictor_rows.append({
        "feature": column,
        "count": series.count(),
        "missing_pct": series.isna().mean() * 100,
        "zero_pct": (series == 0).mean() * 100,
        "mean": series.mean(),
        "std": series.std(),
        "variance": series.var(),
        "min": series.min(),
        "p01": series.quantile(0.01),
        "p02": series.quantile(0.02),
        "p25": q25,
        "median": series.median(),
        "p75": q75,
        "p98": series.quantile(0.98),
        "p99": series.quantile(0.99),
        "max": series.max(),
        "IQR": iqr,
        "skewness": series.skew(),
        "unique_n": series.nunique(dropna=True),
    })

predictor_summary = pd.DataFrame(
    predictor_rows
)

# Scale-free near-constant indicator:
# flag only variables with <= 2 distinct values or zero IQR.
predictor_summary["near_constant_flag"] = (
    (predictor_summary["unique_n"] <= 2)
    | (predictor_summary["IQR"] == 0)
)

display(
    predictor_summary
    .sort_values(
        "skewness",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .round(3)
)

predictor_summary.to_csv(
    TABLE_DIR
    / "predictor_distribution_summary.csv",
    index=False,
)

print(
    "Near-constant predictors:",
    predictor_summary.loc[
        predictor_summary["near_constant_flag"],
        "feature",
    ].tolist(),
)

In [ ]:
# Plot every predictor in small multiples for diagnostic review.
# This figure is intended mainly for notebook/appendix use.

n_features = len(BASELINE_FEATURES)
n_cols = 4
n_rows = int(
    np.ceil(
        n_features / n_cols
    )
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 2.8 * n_rows),
)

axes = np.array(
    axes
).reshape(-1)

for ax, column in zip(
    axes,
    BASELINE_FEATURES,
):
    values = (
        pd.to_numeric(
            df[column],
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    sns.histplot(
        values,
        bins=30,
        ax=ax,
    )

    ax.set_title(
        column,
        fontsize=9,
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

for ax in axes[
    len(BASELINE_FEATURES):
]:
    ax.set_visible(False)

fig.suptitle(
    "Distribution of all tabular predictors",
    fontsize=14,
    y=1.002,
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "all_predictor_distributions.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

## 9. Feature correlation

Pearson correlation captures linear association; Spearman correlation is also reported because many urban variables are skewed and relationships may be monotonic but non-linear.

In [ ]:
analysis_columns = (
    BASELINE_FEATURES
    + [TARGET_COLUMN]
)

correlation_data = (
    df[
        analysis_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

pearson_corr = correlation_data.corr(
    method="pearson"
)

spearman_corr = correlation_data.corr(
    method="spearman"
)

target_correlations = pd.DataFrame({
    "feature": BASELINE_FEATURES,
    "pearson_r": [
        pearson_corr.loc[
            feature,
            TARGET_COLUMN,
        ]
        for feature in BASELINE_FEATURES
    ],
    "spearman_rho": [
        spearman_corr.loc[
            feature,
            TARGET_COLUMN,
        ]
        for feature in BASELINE_FEATURES
    ],
})

target_correlations["abs_spearman_rho"] = (
    target_correlations[
        "spearman_rho"
    ].abs()
)

target_correlations = (
    target_correlations
    .sort_values(
        "abs_spearman_rho",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    target_correlations
    .head(15)
    .round(3)
)

target_correlations.to_csv(
    TABLE_DIR
    / "feature_target_correlations.csv",
    index=False,
)

pearson_corr.to_csv(
    TABLE_DIR
    / "pearson_correlation_matrix.csv"
)

spearman_corr.to_csv(
    TABLE_DIR
    / "spearman_correlation_matrix.csv"
)

In [ ]:
top_features = (
    target_correlations
    .head(15)
    .sort_values(
        "spearman_rho"
    )
)

fig, ax = plt.subplots(
    figsize=(8, 6),
)

ax.barh(
    top_features["feature"],
    top_features["spearman_rho"],
)

ax.axvline(
    0,
    linewidth=0.8,
)

ax.set_title(
    "Top 15 predictor associations with electricity consumption"
)

ax.set_xlabel(
    "Spearman correlation"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "top15_feature_target_spearman.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Full Spearman correlation heatmap for diagnostic review.

fig, ax = plt.subplots(
    figsize=(18, 15),
)

sns.heatmap(
    spearman_corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=False,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={
        "label": "Spearman correlation"
    },
    ax=ax,
)

ax.set_title(
    "Spearman correlation matrix"
)

ax.tick_params(
    axis="x",
    labelrotation=90,
    labelsize=7,
)

ax.tick_params(
    axis="y",
    labelsize=7,
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "spearman_correlation_heatmap_full.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 10. Representative feature–target relationships

A small set of interpretable predictors is shown rather than plotting all 37 variables against the target in the dissertation.

In [ ]:
candidate_features = [
    "NTL_max",
    "pop_density_km2",
    "building_count",
    "building_coverage",
    "road_density",
    "poi_economic_count",
]

relationship_features = [
    column
    for column in candidate_features
    if column in df.columns
]

n_cols = 3
n_rows = int(
    np.ceil(
        len(relationship_features)
        / n_cols
    )
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4.5 * n_rows),
)

axes = np.array(
    axes
).reshape(-1)

for ax, feature in zip(
    axes,
    relationship_features,
):
    ax.scatter(
        df[feature],
        df[TARGET_COLUMN],
        s=15,
        alpha=0.35,
    )

    ax.set_title(
        feature
    )

    ax.set_xlabel(
        feature
    )

    ax.set_ylabel(
        TARGET_COLUMN
    )

for ax in axes[
    len(relationship_features):
]:
    ax.set_visible(False)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "representative_feature_target_scatter.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Inspect spatial locations of missing NTL values
# ============================================================

NTL_COLUMNS = [
    "NTL_mean",
    "NTL_std",
    "NTL_max",
]

ntl_missing_mask = (
    eda_gdf[NTL_COLUMNS]
    .isna()
    .any(axis=1)
)

ntl_missing_gdf = (
    eda_gdf.loc[
        ntl_missing_mask,
        [
            "grid_id",
            TARGET_COLUMN,
            *NTL_COLUMNS,
            "geometry",
        ],
    ]
    .copy()
)

print(
    "Grids with any missing NTL:",
    len(ntl_missing_gdf),
)

print("\nMissing count by variable:")
print(
    eda_gdf[
        NTL_COLUMNS
    ]
    .isna()
    .sum()
)

display(
    ntl_missing_gdf.drop(
        columns="geometry"
    )
)

fig, ax = plt.subplots(
    figsize=(8, 8)
)

eda_gdf.boundary.plot(
    ax=ax,
    color="lightgrey",
    linewidth=0.25,
)

ntl_missing_gdf.plot(
    ax=ax,
    color="red",
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title(
    "Locations of grids with missing VIIRS NTL values"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Diagnose NTL_std information content
# ============================================================

ntl_diagnostic = []

for column in [
    "NTL_mean",
    "NTL_std",
    "NTL_max",
]:
    
    s = eda_gdf[
        column
    ]

    ntl_diagnostic.append({
        "feature": column,
        "missing_pct": (
            s.isna().mean() * 100
        ),
        "zero_pct": (
            (s == 0).mean() * 100
        ),
        "unique_n": (
            s.nunique(
                dropna=True
            )
        ),
        "mean": s.mean(),
        "std": s.std(),
        "IQR": (
            s.quantile(0.75)
            - s.quantile(0.25)
        ),
        "skewness": s.skew(),
        "spearman_with_target": (
            eda_gdf[
                [column, TARGET_COLUMN]
            ]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        ),
    })

ntl_diagnostic = pd.DataFrame(
    ntl_diagnostic
)

display(
    ntl_diagnostic.round(3)
)

In [ ]:
# ============================================================
# Diagnose clipped boundary grids using area_m2
# ============================================================

FULL_GRID_AREA = 250000

area_check = (
    eda_gdf[
        [
            "grid_id",
            "area_m2",
            TARGET_COLUMN,
            "geometry",
        ]
    ]
    .copy()
)

area_check[
    "area_ratio"
] = (
    area_check["area_m2"]
    / FULL_GRID_AREA
)

print(
    area_check[
        "area_ratio"
    ]
    .describe(
        percentiles=[
            0.01,
            0.02,
            0.05,
            0.10,
        ]
    )
)

print(
    "\nSpearman area vs electricity:",
    area_check[
        [
            "area_m2",
            TARGET_COLUMN,
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[0, 1]
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

axes[0].scatter(
    area_check["area_ratio"],
    area_check[TARGET_COLUMN],
    s=15,
    alpha=0.35,
)

axes[0].set_xlabel(
    "Grid area / full 500 m grid area"
)

axes[0].set_ylabel(
    "Electricity consumption"
)

axes[0].set_title(
    "Grid area versus electricity consumption"
)

area_check.plot(
    column="area_ratio",
    ax=axes[1],
    cmap="viridis",
    legend=True,
    edgecolor="none",
)

axes[1].set_title(
    "Spatial distribution of clipped grid area"
)

axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Inspect extreme Sentinel maximum values
# ============================================================

SENTINEL_MAX_COLUMNS = [
    "B02_max",
    "B03_max",
    "B04_max",
    "B08_max",
    "B11_max",
]

extreme_frames = []

for column in (
    SENTINEL_MAX_COLUMNS
):
    
    threshold = (
        eda_gdf[column]
        .quantile(0.99)
    )

    subset = (
        eda_gdf.loc[
            eda_gdf[column]
            > threshold,
            [
                "grid_id",
                column,
                TARGET_COLUMN,
                "geometry",
            ],
        ]
        .copy()
    )

    subset[
        "feature"
    ] = column

    subset[
        "p99_threshold"
    ] = threshold

    subset = subset.rename(
        columns={
            column:
            "feature_value"
        }
    )

    extreme_frames.append(
        subset
    )

sentinel_extremes = pd.concat(
    extreme_frames,
    ignore_index=True,
)

display(
    sentinel_extremes[
        [
            "grid_id",
            "feature",
            "feature_value",
            "p99_threshold",
            TARGET_COLUMN,
        ]
    ]
    .sort_values(
        [
            "feature",
            "feature_value",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

fig, ax = plt.subplots(
    figsize=(8, 8)
)

eda_gdf.boundary.plot(
    ax=ax,
    color="lightgrey",
    linewidth=0.2,
)

gpd.GeoDataFrame(
    sentinel_extremes,
    geometry="geometry",
    crs=eda_gdf.crs,
).plot(
    ax=ax,
    color="red",
    edgecolor="black",
    linewidth=0.3,
)

ax.set_title(
    "Grids exceeding the 99th percentile of Sentinel maximum features"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Check clipped-grid sizes and overlap with missing NTL
# ============================================================

FULL_GRID_AREA = 250000

eda_gdf["area_ratio"] = (
    eda_gdf["area_m2"]
    / FULL_GRID_AREA
)

thresholds = [
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
]

rows = []

for threshold in thresholds:

    small_mask = (
        eda_gdf["area_ratio"]
        < threshold
    )

    rows.append({
        "area_ratio_threshold": threshold,
        "area_m2_threshold": (
            FULL_GRID_AREA
            * threshold
        ),
        "n_grids_below": (
            small_mask.sum()
        ),
        "pct_grids_below": (
            small_mask.mean()
            * 100
        ),
        "missing_NTL_inside": (
            (
                small_mask
                & ntl_missing_mask
            )
            .sum()
        ),
    })

area_threshold_table = pd.DataFrame(
    rows
)

display(
    area_threshold_table
    .round(2)
)

print(
    "\nArea ratios of the 11 missing-NTL grids:"
)

display(
    eda_gdf.loc[
        ntl_missing_mask,
        [
            "grid_id",
            "area_m2",
            "area_ratio",
            TARGET_COLUMN,
        ],
    ]
    .sort_values(
        "area_ratio"
    )
)

In [ ]:
# ============================================================
# Map unique Sentinel extreme grids
# ============================================================

extreme_grid_ids = (
    sentinel_extremes[
        "grid_id"
    ]
    .drop_duplicates()
)

extreme_grid_gdf = (
    eda_gdf[
        eda_gdf["grid_id"]
        .isin(extreme_grid_ids)
    ]
    .copy()
)

print(
    "Unique extreme grids:",
    len(extreme_grid_gdf)
)

fig, ax = plt.subplots(
    figsize=(10, 9)
)

eda_gdf.boundary.plot(
    ax=ax,
    color="lightgrey",
    linewidth=0.25,
)

extreme_grid_gdf.plot(
    ax=ax,
    facecolor="red",
    edgecolor="black",
    alpha=0.65,
)

for _, row in (
    extreme_grid_gdf
    .iterrows()
):
    point = (
        row.geometry.centroid
    )

    ax.text(
        point.x,
        point.y,
        str(row["grid_id"]),
        fontsize=6,
        ha="center",
        va="center",
    )

ax.set_title(
    "Sentinel extreme-value grids"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Display RGB patches for the most extreme Sentinel grids
# ============================================================

import rasterio
from rasterio.mask import mask as rio_mask

TCI_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "TCI_k.tif"

# Rank each grid by its strongest exceedance above p99
extreme_rank = (
    sentinel_extremes
    .assign(
        exceedance_ratio=(
            sentinel_extremes[
                "feature_value"
            ]
            / sentinel_extremes[
                "p99_threshold"
            ]
        )
    )
    .groupby(
        "grid_id",
        as_index=False,
    )
    .agg(
        strongest_ratio=(
            "exceedance_ratio",
            "max",
        ),
        strongest_value=(
            "feature_value",
            "max",
        ),
    )
    .sort_values(
        "strongest_ratio",
        ascending=False,
    )
)

# Start by inspecting the 12 most extreme unique grids
TOP_N = 12

inspect_ids = (
    extreme_rank[
        "grid_id"
    ]
    .head(TOP_N)
    .tolist()
)

inspect_gdf = (
    eda_gdf[
        eda_gdf["grid_id"]
        .isin(inspect_ids)
    ]
    .copy()
)

with rasterio.open(
    TCI_PATH
) as src:

    # Ensure geometries use the raster CRS
    inspect_gdf_raster = (
        inspect_gdf
        .to_crs(src.crs)
    )

    n_cols = 4
    n_rows = int(
        np.ceil(
            len(inspect_ids)
            / n_cols
        )
    )

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(
            14,
            3.5 * n_rows,
        ),
    )

    axes = np.array(
        axes
    ).reshape(-1)

    for ax, grid_id in zip(
        axes,
        inspect_ids,
    ):

        row = (
            inspect_gdf_raster[
                inspect_gdf_raster[
                    "grid_id"
                ]
                == grid_id
            ]
            .iloc[0]
        )

        image, _ = rio_mask(
            src,
            [
                row.geometry
                .__geo_interface__
            ],
            crop=True,
            filled=True,
            indexes=[
                1,
                2,
                3,
            ],
        )

        rgb = np.moveaxis(
            image,
            0,
            -1,
        )

        ax.imshow(
            rgb
        )

        # Which Sentinel feature(s) were extreme?
        extreme_features = (
            sentinel_extremes.loc[
                sentinel_extremes[
                    "grid_id"
                ]
                == grid_id,
                "feature",
            ]
            .unique()
        )

        feature_text = ", ".join(
            extreme_features
        )

        ax.set_title(
            f"Grid {grid_id}\\n{feature_text}",
            fontsize=9,
        )

        ax.set_axis_off()

    for ax in axes[
        len(inspect_ids):
    ]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

## 17. Candidate multicollinearity diagnostics

Correlation and VIF are used as diagnostics, not as automatic feature-selection rules. Tree-based models can tolerate correlated predictors; variables are removed only where a clear deterministic redundancy or low-information argument exists.

In [ ]:
def multicollinearity_diagnostics(frame, features, prefix, threshold=0.90):
    X = frame[features].replace([np.inf, -np.inf], np.nan).copy()

    constant_features = [
        column for column in X.columns
        if X[column].nunique(dropna=True) <= 1
    ]

    if constant_features:
        print(f"{prefix}: excluding constants from VIF:", constant_features)
        X = X.drop(columns=constant_features)

    imputer = SimpleImputer(strategy="median")
    X_imputed = pd.DataFrame(
        imputer.fit_transform(X),
        columns=X.columns,
        index=X.index,
    )

    corr = X_imputed.corr(method="pearson")
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    high_pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
    )
    high_pairs["abs_correlation"] = high_pairs["correlation"].abs()
    high_pairs = (
        high_pairs[high_pairs["abs_correlation"] >= threshold]
        .sort_values("abs_correlation", ascending=False)
        .reset_index(drop=True)
    )

    vif_values = []
    values = X_imputed.to_numpy(dtype=float)
    for i, feature in enumerate(X_imputed.columns):
        try:
            vif_value = variance_inflation_factor(values, i)
        except Exception:
            vif_value = np.nan
        vif_values.append(vif_value)

    vif_table = (
        pd.DataFrame({"feature": X_imputed.columns, "VIF": vif_values})
        .sort_values("VIF", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

    high_pairs.to_csv(TABLE_DIR / f"{prefix}_high_correlation_pairs.csv", index=False)
    vif_table.to_csv(TABLE_DIR / f"{prefix}_vif.csv", index=False)

    return corr, high_pairs, vif_table

candidate_corr, candidate_high_pairs, candidate_vif = multicollinearity_diagnostics(
    df,
    CANDIDATE_FEATURES,
    prefix="candidate",
)

print("Highly correlated candidate pairs:", len(candidate_high_pairs))
display(candidate_high_pairs.head(30).round(3))
display(candidate_vif.head(30).round(2))

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(candidate_corr, dtype=bool))
sns.heatmap(
    candidate_corr,
    mask=mask,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.1,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Pearson correlation matrix — 42 candidate predictors")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "candidate_predictor_pearson_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Feature audit and final 36 predictors

Six variables are removed after EDA and feature audit:

- `brightness`: deterministic aggregate of retained Sentinel mean bands;
- `poi_total_count`: deterministic sum of the three retained POI category counts;
- `pop_total`: redundant with population density and effective grid area;
- `road_length_total`: redundant with road density and effective grid area;
- `building_area_total`: redundant with building coverage and effective grid area;
- `NTL_std`: retained during EDA for diagnosis, then removed because it is highly zero-inflated / low-information relative to `NTL_mean` and `NTL_max`.

`area_m2` is retained as the effective clipped grid-area exposure variable.

In [ ]:
feature_audit_table = pd.DataFrame([
    {"feature": "brightness", "decision": "drop", "reason": "Deterministic aggregate of retained Sentinel mean bands."},
    {"feature": "poi_total_count", "decision": "drop", "reason": "Deterministic sum of retained POI category counts."},
    {"feature": "pop_total", "decision": "drop", "reason": "Redundant with population density and effective grid area."},
    {"feature": "road_length_total", "decision": "drop", "reason": "Redundant with road density and effective grid area."},
    {"feature": "building_area_total", "decision": "drop", "reason": "Redundant with building coverage and effective grid area."},
    {"feature": "NTL_std", "decision": "drop", "reason": "Highly zero-inflated / low-information NTL summary; retain NTL_mean and NTL_max."},
])

feature_audit_table.to_csv(TABLE_DIR / "feature_audit_decisions.csv", index=False)
display(feature_audit_table)

final_df = (
    df[META_COLUMNS + FINAL_FEATURES]
    .copy()
    .sort_values("grid_id")
    .reset_index(drop=True)
)

if len(final_df) != len(df):
    raise ValueError("Feature cleaning changed the number of rows.")
if not final_df["grid_id"].is_unique:
    raise ValueError("grid_id is not unique in the final matrix.")
if final_df[TARGET_COLUMN].isna().any():
    raise ValueError("Target contains missing values in the final matrix.")
if set(FEATURES_TO_DROP).intersection(final_df.columns):
    raise ValueError("A dropped feature remains in the final matrix.")

# Intentionally overwrite the original path so 002 keeps the same input path.
final_df.to_csv(FEATURE_MATRIX_PATH, index=False)

print("Final matrix saved:", FEATURE_MATRIX_PATH)
print("Rows:", len(final_df))
print("Final feature count:", len(FINAL_FEATURES))
print("Total columns including metadata/target:", final_df.shape[1])

## 19. Final 36-feature diagnostics

In [ ]:
final_predictor_summary = []
for column in FINAL_FEATURES:
    series = pd.to_numeric(final_df[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
    final_predictor_summary.append({
        "feature": column,
        "missing_pct": series.isna().mean() * 100,
        "zero_pct": (series == 0).mean() * 100,
        "mean": series.mean(),
        "std": series.std(),
        "median": series.median(),
        "skewness": series.skew(),
        "unique_n": series.nunique(dropna=True),
    })

final_predictor_summary = pd.DataFrame(final_predictor_summary)
final_predictor_summary.to_csv(TABLE_DIR / "final_36_predictor_summary.csv", index=False)
display(final_predictor_summary.round(3))

final_corr, final_high_pairs, final_vif = multicollinearity_diagnostics(
    final_df,
    FINAL_FEATURES,
    prefix="final",
)

display(final_high_pairs.head(30).round(3))
display(final_vif.head(30).round(2))

fig, ax = plt.subplots(figsize=(15, 13))
mask = np.triu(np.ones_like(final_corr, dtype=bool))
sns.heatmap(
    final_corr,
    mask=mask,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.1,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Pearson correlation matrix — final 36 predictors")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_36_predictor_pearson_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
feature_family_rows = []
for feature in FINAL_FEATURES:
    if feature == "area_m2":
        family = "Exposure / grid area"
    elif feature.startswith(("B02_", "B03_", "B04_", "B08_", "B11_")) or feature in ["NDVI", "NDBI"]:
        family = "Sentinel-2"
    elif feature.startswith("NTL_"):
        family = "VIIRS nighttime lights"
    elif feature == "dist_major_road" or feature.startswith("road_") or feature == "major_road_ratio":
        family = "Road network"
    elif feature.startswith("poi_") or (feature.startswith("dist_") and feature.endswith("_poi")):
        family = "POI"
    elif feature.startswith("pop_"):
        family = "WorldPop"
    elif feature.startswith("building_"):
        family = "Building morphology"
    else:
        family = "Other"
    feature_family_rows.append({"feature": feature, "family": family})

final_feature_inventory = pd.DataFrame(feature_family_rows)
final_feature_inventory.to_csv(TABLE_DIR / "final_feature_inventory.csv", index=False)
display(final_feature_inventory)
display(final_feature_inventory["family"].value_counts().rename_axis("family").reset_index(name="n_features"))

# Part B — CNN visual representation and backbone screening

The tabular matrix is now fixed at 36 features. CNN extraction is kept separate so there is no ambiguity about the baseline.

The visual branch uses Sentinel-2 B04/B03/B02 true-colour patches, ImageNet normalisation and frozen pretrained ResNet backbones. Raw embeddings are saved separately; PCA is fitted **inside each training fold** during validation.

In [ ]:
BAND_PATHS = [
    RAW_DATA_DIR / "daylight21" / "processing" / "B04_k.tif",
    RAW_DATA_DIR / "daylight21" / "processing" / "B03_k.tif",
    RAW_DATA_DIR / "daylight21" / "processing" / "B02_k.tif",
]

CNN_OUTPUT_PATHS = {
    "ResNet18": FEATURE_DIR / "cnn_features_500m_resnet18_raw.csv",
    "ResNet34": FEATURE_DIR / "cnn_features_500m_resnet34_imagenet_raw.csv",
    "ResNet50": FEATURE_DIR / "cnn_features_500m_resnet50_raw.csv",
}

for path in BAND_PATHS:
    if not path.exists():
        raise FileNotFoundError(f"Sentinel RGB band not found: {path}")

imagenet_transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class SatelliteGridDataset(Dataset):
    # Crop the three Sentinel-2 RGB bands to each grid polygon.
    def __init__(self, gdf, band_paths, transform=None):
        self.gdf = gdf.reset_index(drop=True).copy()
        self.band_paths = [Path(path) for path in band_paths]
        self.transform = transform

        with rasterio.open(self.band_paths[0]) as src:
            raster_crs = src.crs

        if self.gdf.crs != raster_crs:
            self.gdf = self.gdf.to_crs(raster_crs)

    def __len__(self):
        return len(self.gdf)

    def __getitem__(self, idx):
        row = self.gdf.iloc[idx]
        geometry = [row.geometry.__geo_interface__]
        channels = []

        for band_path in self.band_paths:
            with rasterio.open(band_path) as src:
                image, _ = rio_mask(src, geometry, crop=True, filled=True)
            array = np.nan_to_num(image[0], nan=0.0, posinf=0.0, neginf=0.0)
            channels.append(array)

        image = np.stack(channels, axis=0)
        tensor = torch.as_tensor(image, dtype=torch.float32) / 10000.0
        tensor = tensor.clamp(0.0, 1.0)

        if self.transform is not None:
            tensor = self.transform(tensor)

        return tensor, int(row["grid_id"])

In [ ]:
def build_feature_extractor(backbone_name):
    if backbone_name == "ResNet18":
        backbone = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        expected_dimension = 512
    elif backbone_name == "ResNet34":
        backbone = models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        expected_dimension = 512
    elif backbone_name == "ResNet50":
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        expected_dimension = 2048
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    feature_extractor = torch.nn.Sequential(*list(backbone.children())[:-1]).to(DEVICE).eval()
    for parameter in feature_extractor.parameters():
        parameter.requires_grad = False

    return feature_extractor, expected_dimension


def extract_backbone_features(grid_frame, backbone_name, batch_size=64):
    dataset = SatelliteGridDataset(grid_frame, BAND_PATHS, transform=imagenet_transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    extractor, expected_dimension = build_feature_extractor(backbone_name)

    feature_batches = []
    grid_ids = []

    print(f"Extracting {backbone_name} features...")
    with torch.no_grad():
        for images, batch_grid_ids in dataloader:
            images = images.to(DEVICE)
            features = extractor(images).flatten(1)
            feature_batches.append(features.cpu().numpy())
            grid_ids.extend(batch_grid_ids.cpu().numpy().astype(int).tolist())

    feature_array = np.vstack(feature_batches)

    if feature_array.shape[1] != expected_dimension:
        raise ValueError(
            f"{backbone_name}: expected {expected_dimension} dimensions, "
            f"found {feature_array.shape[1]}."
        )
    if not np.isfinite(feature_array).all():
        raise ValueError(f"{backbone_name}: non-finite visual features found.")

    return np.asarray(grid_ids, dtype=int), feature_array

## 20. Extract or reuse ImageNet ResNet18 / ResNet34 / ResNet50 embeddings

In [ ]:
FORCE_CNN_REEXTRACTION = False
CNN_BATCH_SIZE = 64

cnn_grid = (
    grid_gdf[grid_gdf["grid_id"].isin(final_df["grid_id"])]
    .copy()
    .sort_values("grid_id")
    .reset_index(drop=True)
)

if len(cnn_grid) != len(final_df):
    raise ValueError("CNN grid count does not match final tabular matrix.")

backbone_dimensions = {"ResNet18": 512, "ResNet34": 512, "ResNet50": 2048}
cnn_tables = {}

for backbone_name, expected_dimension in backbone_dimensions.items():
    output_path = CNN_OUTPUT_PATHS[backbone_name]
    use_cache = False

    if output_path.exists() and not FORCE_CNN_REEXTRACTION:
        cached = pd.read_csv(output_path)
        cached_features = [column for column in cached.columns if column.startswith("cnn_")]
        cache_valid = (
            "grid_id" in cached.columns
            and cached["grid_id"].is_unique
            and len(cached) == len(final_df)
            and len(cached_features) == expected_dimension
            and set(cached["grid_id"]) == set(final_df["grid_id"])
        )
        if cache_valid:
            cnn_tables[backbone_name] = cached.sort_values("grid_id").reset_index(drop=True)
            use_cache = True
            print(f"{backbone_name}: using cached file -> {output_path}")

    if use_cache:
        continue

    grid_ids, feature_array = extract_backbone_features(
        cnn_grid,
        backbone_name,
        batch_size=CNN_BATCH_SIZE,
    )

    feature_columns = [f"cnn_{i}" for i in range(expected_dimension)]
    cnn_table = pd.DataFrame(feature_array, columns=feature_columns)
    cnn_table.insert(0, "grid_id", grid_ids)
    cnn_table = cnn_table.sort_values("grid_id").reset_index(drop=True)

    if not cnn_table["grid_id"].is_unique:
        raise ValueError(f"{backbone_name}: duplicate grid_id values.")
    if set(cnn_table["grid_id"]) != set(final_df["grid_id"]):
        raise ValueError(f"{backbone_name}: grid_id mismatch with final matrix.")

    cnn_table.to_csv(output_path, index=False)
    cnn_tables[backbone_name] = cnn_table
    print(f"{backbone_name}: saved -> {output_path}")

print("ResNet50 downstream file:", CNN_OUTPUT_PATHS["ResNet50"])

# Part C — CNN backbone screening

This is architecture screening, not final hyperparameter tuning. The tabular-only configuration and each CNN configuration use the same 36-feature baseline. CNN PCA is fitted separately within every training fold and retains 80% explained variance. KMeans K=4 is used only for the selected 500 m dataset; the final shared spatial-block output remains the responsibility of `02_model_screening_and_tuning.ipynb`.

In [ ]:
RANDOM_FOLDS = 5
N_SPATIAL_CLUSTERS = 4
PCA_VARIANCE_THRESHOLD = 0.80

screening_df = final_df.sort_values("grid_id").reset_index(drop=True).copy()

kmeans = KMeans(n_clusters=N_SPATIAL_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
screening_df["spatial_block"] = kmeans.fit_predict(
    screening_df[["centroid_x", "centroid_y"]].to_numpy()
).astype(int)

screening_df[["grid_id", "centroid_x", "centroid_y", "spatial_block"]].to_csv(
    MODEL_RESULT_DIR / "backbone_screening_spatial_blocks.csv",
    index=False,
)

print(screening_df["spatial_block"].value_counts().sort_index())

In [ ]:
spatial_palette = sns.color_palette("Set2", n_colors=N_SPATIAL_CLUSTERS)
spatial_cmap = ListedColormap(spatial_palette)

screening_grid = (
    grid_gdf[["grid_id", "geometry"]]
    .merge(
        screening_df[["grid_id", "spatial_block"]],
        on="grid_id",
        how="inner",
        validate="one_to_one",
    )
)

fig, ax = plt.subplots(figsize=(8, 8))
screening_grid.plot(
    column="spatial_block",
    cmap=spatial_cmap,
    vmin=0,
    vmax=N_SPATIAL_CLUSTERS - 1,
    edgecolor="white",
    linewidth=0.25,
    ax=ax,
)
legend_handles = [
    mpatches.Patch(color=spatial_palette[i], label=f"Cluster {i}")
    for i in range(N_SPATIAL_CLUSTERS)
]
ax.legend(handles=legend_handles, title="Spatial fold", loc="lower left")
ax.set_title("KMeans K=4 spatial folds for CNN backbone screening")
ax.set_axis_off()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_backbone_screening_kmeans_folds.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
MODELS = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    ),
}

random_cv = KFold(n_splits=RANDOM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
random_splits = [
    {
        "fold": fold,
        "split_id": fold,
        "train_index": train_idx,
        "test_index": test_idx,
    }
    for fold, (train_idx, test_idx) in enumerate(random_cv.split(screening_df), start=1)
]

spatial_splits = []
for fold, block in enumerate(sorted(screening_df["spatial_block"].unique()), start=1):
    test_idx = np.flatnonzero(screening_df["spatial_block"].to_numpy() == block)
    train_idx = np.flatnonzero(screening_df["spatial_block"].to_numpy() != block)
    spatial_splits.append({
        "fold": fold,
        "split_id": int(block),
        "train_index": train_idx,
        "test_index": test_idx,
    })

VALIDATION_SCHEMES = {
    "random_5fold": random_splits,
    "spatial_kmeans_leave_one_out": spatial_splits,
}

for name, splits in VALIDATION_SCHEMES.items():
    print(name, "test sizes:", [len(split["test_index"]) for split in splits])

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


def prepare_fold_features(train_frame, test_frame, cnn_table=None):
    tabular_imputer = SimpleImputer(strategy="median")
    X_train_tab = tabular_imputer.fit_transform(train_frame[FINAL_FEATURES])
    X_test_tab = tabular_imputer.transform(test_frame[FINAL_FEATURES])

    if cnn_table is None:
        return X_train_tab, X_test_tab, 0

    cnn_columns = [column for column in cnn_table.columns if column.startswith("cnn_")]
    train_cnn = train_frame[["grid_id"]].merge(cnn_table, on="grid_id", how="left", validate="one_to_one")
    test_cnn = test_frame[["grid_id"]].merge(cnn_table, on="grid_id", how="left", validate="one_to_one")

    cnn_imputer = SimpleImputer(strategy="median")
    X_train_cnn = cnn_imputer.fit_transform(train_cnn[cnn_columns])
    X_test_cnn = cnn_imputer.transform(test_cnn[cnn_columns])

    pca = PCA(n_components=PCA_VARIANCE_THRESHOLD, svd_solver="full")
    X_train_pca = pca.fit_transform(X_train_cnn)
    X_test_pca = pca.transform(X_test_cnn)

    return (
        np.hstack([X_train_tab, X_train_pca]),
        np.hstack([X_test_tab, X_test_pca]),
        int(pca.n_components_),
    )


def evaluate_configuration(frame, splits, validation_name, feature_set, model_name, model_template):
    cnn_table = None if feature_set == "Tabular" else cnn_tables[feature_set]
    fold_rows = []
    prediction_frames = []

    for split in splits:
        fold = split["fold"]
        train_idx = split["train_index"]
        test_idx = split["test_index"]
        train_frame = frame.iloc[train_idx].copy()
        test_frame = frame.iloc[test_idx].copy()

        y_train = train_frame[TARGET_COLUMN].to_numpy(dtype=float)
        y_test = test_frame[TARGET_COLUMN].to_numpy(dtype=float)

        X_train, X_test, n_pca = prepare_fold_features(train_frame, test_frame, cnn_table)
        model = clone(model_template)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        metrics = regression_metrics(y_test, test_pred)

        fold_rows.append({
            "validation": validation_name,
            "feature_set": feature_set,
            "model": model_name,
            "fold": fold,
            "split_id": split["split_id"],
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "R2": metrics["R2"],
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "train_R2": r2_score(y_train, train_pred),
            "cnn_pca_components": n_pca,
        })

        prediction_frames.append(pd.DataFrame({
            "grid_id": test_frame["grid_id"].to_numpy(),
            "validation": validation_name,
            "feature_set": feature_set,
            "model": model_name,
            "fold": fold,
            "observed": y_test,
            "predicted": test_pred,
        }))

    return pd.DataFrame(fold_rows), pd.concat(prediction_frames, ignore_index=True)

## 21. Run the backbone screening and save outputs

In [ ]:
SCREENING_FEATURE_SETS = ["Tabular", "ResNet18", "ResNet34", "ResNet50"]
all_fold_results = []
all_predictions = []

for validation_name, splits in VALIDATION_SCHEMES.items():
    for model_name, model_template in MODELS.items():
        for feature_set in SCREENING_FEATURE_SETS:
            print(f"{validation_name} | {model_name} | {feature_set}")
            fold_result, predictions = evaluate_configuration(
                screening_df,
                splits,
                validation_name,
                feature_set,
                model_name,
                model_template,
            )
            all_fold_results.append(fold_result)
            all_predictions.append(predictions)

fold_results = pd.concat(all_fold_results, ignore_index=True)
oof_predictions = pd.concat(all_predictions, ignore_index=True)

summary = (
    fold_results
    .groupby(["validation", "feature_set", "model"], as_index=False)
    .agg(
        n_folds=("fold", "count"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", lambda x: x.std(ddof=0)),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", lambda x: x.std(ddof=0)),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", lambda x: x.std(ddof=0)),
        train_R2_mean=("train_R2", "mean"),
        cnn_pca_components_mean=("cnn_pca_components", "mean"),
    )
)

pooled_rows = []
for (validation, feature_set, model), group in oof_predictions.groupby(["validation", "feature_set", "model"]):
    metrics = regression_metrics(group["observed"].to_numpy(), group["predicted"].to_numpy())
    pooled_rows.append({
        "validation": validation,
        "feature_set": feature_set,
        "model": model,
        "OOF_R2": metrics["R2"],
        "OOF_RMSE": metrics["RMSE"],
        "OOF_MAE": metrics["MAE"],
    })

summary = summary.merge(
    pd.DataFrame(pooled_rows),
    on=["validation", "feature_set", "model"],
    how="left",
    validate="one_to_one",
)

fold_results.to_csv(MODEL_RESULT_DIR / "fold_results.csv", index=False)
oof_predictions.to_csv(MODEL_RESULT_DIR / "oof_predictions.csv", index=False)
summary.to_csv(MODEL_RESULT_DIR / "model_summary.csv", index=False)

display(summary.sort_values(["validation", "R2_mean"], ascending=[True, False]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
validation_titles = {
    "random_5fold": "Random 5-fold CV",
    "spatial_kmeans_leave_one_out": "KMeans K=4 spatial CV",
}

for ax, validation_name in zip(axes, validation_titles):
    plot_data = summary[
        (summary["validation"] == validation_name)
        & (summary["model"] == "XGBoost")
    ].copy()

    sns.barplot(data=plot_data, x="feature_set", y="R2_mean", ax=ax)
    ax.axhline(0, linewidth=0.8)
    ax.set_title(validation_titles[validation_name])
    ax.set_xlabel("")
    ax.set_ylabel("Mean R²")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("CNN backbone screening with fixed XGBoost", y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_backbone_screening_xgboost.png", dpi=300, bbox_inches="tight")
plt.show()

## 22. ResNet50 handoff to the final CNN experiment

The final controlled CNN experiment uses the frozen ImageNet ResNet50 embeddings saved as `work/grid_size_selection/features/cnn_features_500m_resnet50_raw.csv`.

The final engineered matrix remains `work/grid_size_selection/features/feature_matrix_500m.csv`. Notebook 02 reads this matrix for model tuning and creates the shared KMeans spatial assignments. Notebook 03 then combines the same 36 engineered predictors with the ResNet50 embeddings.


In [ ]:
cnn_random_xgb = (
    summary[
        (summary["validation"] == "random_5fold")
        & (summary["model"] == "XGBoost")
        & (summary["feature_set"] != "Tabular")
    ]
    .sort_values("R2_mean", ascending=False)
    .reset_index(drop=True)
)

display(cnn_random_xgb[[
    "feature_set", "R2_mean", "R2_std", "RMSE_mean", "MAE_mean", "OOF_R2"
]].round(4))

screening_leader = cnn_random_xgb.iloc[0]["feature_set"]
print("Random-CV XGBoost CNN leader:", screening_leader)

if screening_leader != "ResNet50":
    print(
        "WARNING: this rerun does not rank ResNet50 first under the fixed screening criterion. "
        "Review the new result before proceeding to the final multimodal experiment."
    )

selected_cnn_path = CNN_OUTPUT_PATHS["ResNet50"]
selected_cnn_df = pd.read_csv(selected_cnn_path)
selected_cnn_features = [column for column in selected_cnn_df.columns if column.startswith("cnn_")]

if len(selected_cnn_features) != 2048:
    raise ValueError("ResNet50 file must contain exactly 2048 cnn_* features.")
if set(selected_cnn_df["grid_id"]) != set(final_df["grid_id"]):
    raise ValueError("ResNet50 grid IDs do not match final tabular matrix.")

handoff_df = pd.read_csv(FEATURE_MATRIX_PATH)
expected_handoff_columns = META_COLUMNS + FINAL_FEATURES

missing_handoff = [column for column in expected_handoff_columns if column not in handoff_df.columns]
extra_handoff = [column for column in handoff_df.columns if column not in expected_handoff_columns]

if missing_handoff:
    raise KeyError(f"002 handoff is missing columns: {missing_handoff}")
if extra_handoff:
    raise ValueError(f"002 handoff contains unexpected columns: {extra_handoff}")
if not handoff_df["grid_id"].is_unique:
    raise ValueError("002 handoff grid_id is not unique.")
if set(FEATURES_TO_DROP).intersection(handoff_df.columns):
    raise ValueError("A dropped feature remains in the 002 handoff.")

print("002 handoff check passed.")
print("Rows:", len(handoff_df))
print("Final tabular predictors:", len(FINAL_FEATURES))
print("Final matrix:", FEATURE_MATRIX_PATH)
print("ResNet50 raw embeddings:", selected_cnn_path)

## 23. Final output inventory

In [ ]:
print("001 completed successfully.")
print("\nMain modelling handoff:")
print(" -", FEATURE_MATRIX_PATH)
print("\nCandidate backup:")
print(" -", CANDIDATE_BACKUP_PATH)
print("\nCNN raw embedding files:")
for backbone_name, path in CNN_OUTPUT_PATHS.items():
    print(f" - {backbone_name}: {path}")
print("\nEDA figures:")
for path in sorted(FIGURE_DIR.glob("*.png")):
    print(" -", path.name)
print("\nEDA / audit tables:")
for path in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", path.name)
print("\nBackbone-screening tables:")
for path in sorted(MODEL_RESULT_DIR.glob("*.csv")):
    print(" -", path.name)